In [1]:
import numpy as np #this lib numerical operation on array
import pandas as pd # this processes data
import matplotlib.pyplot as plt # this is used for plotting figures
import tensorflow as tf # this is used for designing neural networks
from tensorflow.keras import layers, models # this is used to add layers and models in architecture dense flatten convolutional
from pathlib import Path # this is used to access directories

In [2]:
# 1) Load and preprocess MNIST correctly
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data() # this line in importing dataset mnist

# Normalize to [0, 1]
x_train = x_train.astype('float32') / 255.0 # is normalizing image of training 0 to 1
x_test = x_test.astype('float32') / 255.0 # this line is normalizing testing dataset

# Add channel dimension -> (N, 28, 28, 1)
x_train = np.expand_dims(x_train, axis=-1) # increases dimension by 1 axis tell where to add minus one means at end 60000 28 28
x_test = np.expand_dims(x_test, axis=-1) # n

# Hyperparameters
noise_dim = 100 # noisy vector is taking 100
num_examples_to_generate = 16 #
batch_size = 128 # aik line mein 60000 ni ho sakti train
epochs = 10 # time issue that is why we take 10 only

In [3]:
# 2) Build networks
def build_generator(): # a function is defined with zero arguments
    model = models.Sequential([ # model is created from sequential
        layers.Dense(7 * 7 * 256, input_shape=(noise_dim,)),# is adding   input shape 100 and output is 12544
        layers.BatchNormalization(), # normalizaiton is applied 128 examples ko normalize kry ga
        layers.LeakyReLU(), # leaky relu positive to positive negative k liye zero se thora sa chota bna deta h result saary zero ni hoty by default 0.001
        layers.Reshape((7, 7, 256)), # convert into image 7 7 256
        layers.Conv2DTranspose(128, (5, 5), strides=(1, 1), padding='same'), # convolution image ko shrink jab k ye image ko bara krti hai convolution ka oppositie thats why 2d transport 5 by 5 kernel size
        layers.BatchNormalization(),
        layers.LeakyReLU(),
        layers.Conv2DTranspose(64, (5, 5), strides=(2, 2), padding='same'),
        layers.BatchNormalization(),
        layers.LeakyReLU(),
        layers.Conv2DTranspose(1, (5, 5), strides=(2, 2), padding='same', activation='sigmoid') # channels only 1 and activation is sigmoid 28 28 is image size it means 784 neurons that is why sigmoid if tanh then range minus 1 to pos 1
    ])
    return model

def build_discriminator():
    model = models.Sequential([
        layers.Conv2D(64, (5, 5), strides=(2, 2), padding='same', input_shape=[28, 28, 1]),
        layers.LeakyReLU(),
        layers.Dropout(0.3),
        layers.Conv2D(128, (5, 5), strides=(2, 2), padding='same'),
        layers.LeakyReLU(),
        layers.Dropout(0.3),
        layers.Flatten(),
        layers.Dense(1)
    ])
    return model

generator = build_generator() # calling function
discriminator = build_discriminator()

generator_optimizer = tf.keras.optimizers.Adam(1e-4) # it is telling about learning rate  2 optimizer because two model
discriminator_optimizer = tf.keras.optimizers.Adam(1e-4)
cross_entropy = tf.keras.losses.BinaryCrossentropy(from_logits=True) # agar probability chahiye to true rakhna h

In [4]:
def generator_loss(fake_output):
    return cross_entropy(tf.ones_like(fake_output), fake_output)

def discriminator_loss(real_output, fake_output):
    real_loss = cross_entropy(tf.ones_like(real_output), real_output)
    fake_loss = cross_entropy(tf.zeros_like(fake_output), fake_output)
    return real_loss + fake_loss

In [5]:
# 3) Image generation helper
def generate_and_save_images(model, epoch):
    noise = tf.random.normal([num_examples_to_generate, noise_dim])
    generated_images = model(noise, training=False)

    plt.figure(figsize=(4, 4))
    for i in range(generated_images.shape[0]):
        plt.subplot(4, 4, i + 1)
        img = generated_images[i, :, :, 0]
        plt.imshow(img * 255.0, cmap='gray')
        plt.axis('off')
    # Save to a portable path
    save_dir = Path("generated_images")
    save_dir.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(save_dir / f"image_at_epoch_{epoch:04d}.png")
    plt.close()

In [6]:
# 4) Prepare dataset
train_dataset = tf.data.Dataset.from_tensor_slices(x_train)
train_dataset = train_dataset.shuffle(buffer_size=60000).batch(batch_size)

In [7]:
# 5) Training loop
def train(dataset, epochs):
    for epoch in range(epochs):
        for image_batch in dataset:
            curr_batch_size = tf.shape(image_batch)[0]

            # Train discriminator
            noise = tf.random.normal([curr_batch_size, noise_dim])
            with tf.GradientTape() as disc_tape:
                generated_images = generator(noise, training=True)

                real_output = discriminator(image_batch, training=True)
                fake_output = discriminator(generated_images, training=True)

                disc_loss = discriminator_loss(real_output, fake_output)

            grads = disc_tape.gradient(disc_loss, discriminator.trainable_variables)
            discriminator_optimizer.apply_gradients(zip(grads, discriminator.trainable_variables))

            # Train generator
            noise = tf.random.normal([curr_batch_size, noise_dim])
            with tf.GradientTape() as gen_tape:
                generated_images = generator(noise, training=True)
                fake_output = discriminator(generated_images, training=True)
                gen_loss = generator_loss(fake_output)

            grads = gen_tape.gradient(gen_loss, generator.trainable_variables)
            generator_optimizer.apply_gradients(zip(grads, generator.trainable_variables))

        # Logging every epoch
        print(f'Epoch {epoch + 1}, Generator Loss: {gen_loss.numpy():.4f}, Discriminator Loss: {disc_loss.numpy():.4f}')
        generate_and_save_images(generator, epoch + 1)

# Run training
train(train_dataset, epochs)

Epoch 1, Generator Loss: 1.3538, Discriminator Loss: 0.8839
Epoch 2, Generator Loss: 1.1723, Discriminator Loss: 0.9543
Epoch 3, Generator Loss: 1.0608, Discriminator Loss: 1.1219
Epoch 4, Generator Loss: 1.0341, Discriminator Loss: 1.1733
Epoch 5, Generator Loss: 1.0271, Discriminator Loss: 1.5641
Epoch 6, Generator Loss: 1.1196, Discriminator Loss: 1.1909
Epoch 7, Generator Loss: 1.1895, Discriminator Loss: 1.0531
Epoch 8, Generator Loss: 1.2067, Discriminator Loss: 1.0173
Epoch 9, Generator Loss: 1.3116, Discriminator Loss: 0.8906
Epoch 10, Generator Loss: 1.0610, Discriminator Loss: 1.1227
